# Structured Drug Data RAG Pipeline
## End-to-End RAG System — Labs 5-9 Implementation

**Dataset:** 12 drugs with structured pharmaceutical data (10 fields each)  
**Source:** `pharma-rag/data/drugs.json`  

This notebook implements the full RAG pipeline as taught in Labs 5-9:

| Phase | Lab | Topic |
|:------|:----|:------|
| P0 | — | Setup & Configuration |
| P1 | Lab 5 | Text Preprocessing Foundations |
| P2 | Lab 6 | Text Representation & Retrieval |
| P3 | Lab 7 | Embeddings & Semantic Retrieval |
| P4 | Lab 8 | RAG Assembly |
| P5 | Lab 9 | Grounded Generation with Ollama |
| P6 | — | Evaluation & Failure Analysis |

**Pipeline:**  
structured JSON → field-level chunking → preprocessing → sparse/dense/hybrid retrieval → context package → prompt → grounded LLM answer

**Key Rules:**
- No LangChain / LlamaIndex — everything raw
- Preserve negation words and numbers for medical safety
- Every generated answer includes *"This is not medical advice"*
- `documents` → `fit_transform()`, `query` → `transform()` ONLY

---
# Part 0 — Setup & Configuration
---

In [1]:
!pip install -q pandas numpy scikit-learn rank-bm25 sentence-transformers faiss-cpu nltk requests

In [2]:
import json, os, re, math, time, warnings
import numpy as np
import pandas as pd
import nltk
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

# ── Paths & Configuration ──
DRUGS_JSON = os.path.join('pharma-rag', 'data', 'drugs.json')
SAMPLE_N   = 200
RANDOM_SEED = 42
K = 5
EMBED_MODEL_NAME = 'all-MiniLM-L6-v2'
OLLAMA_HOST  = 'http://localhost:11434'
OLLAMA_MODEL = 'deepseek-r1:1.5b'

np.random.seed(RANDOM_SEED)
print('Configuration loaded.')
print(f'  drugs.json  : {os.path.abspath(DRUGS_JSON)}')
print(f'  Embed model : {EMBED_MODEL_NAME}')
print(f'  Ollama model: {OLLAMA_MODEL}')

Configuration loaded.
  drugs.json  : c:\Users\Admin\pharma-rag\pharma-rag\data\drugs.json
  Embed model : all-MiniLM-L6-v2
  Ollama model: deepseek-r1:1.5b


In [3]:
# ── NLTK Downloads ──
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
print('NLTK downloads complete.')

NLTK downloads complete.


---
# Part 1 — Lab 5: Text Preprocessing Foundations
---

## 1.1 Load & Expand Drugs into Field-Level Chunks

Each drug has 7 text fields. We expand into one row per field per drug, giving us
`12 × 7 = 84` document chunks. Each chunk's text is formatted as:  
`{drug_name} — {field_label}: {text}`

In [4]:
DATA_DIR = "pharma-rag/data"
REVIEWS_CSV = "pharma-rag/data/reviews.csv"
DOCS_JSON = "pharma-rag/data/docs.json"

In [5]:
import json
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DRUGS_JSON = DATA_DIR / "drugs.json"

# Load the JSON
with open(DRUGS_JSON, "r", encoding="utf-8") as f:
    drugs_data = json.load(f)

print(f"Loaded {len(drugs_data)} drugs")
print(drugs_data[0]["name"])

Loaded 12 drugs
Paracetamol


In [6]:
import pandas as pd

FIELD_LABELS = [
    "description",
    "mechanism",
    "indications",
    "dosage",
    "side_effects",
    "contraindications",
    "interactions",
]

records = []

for drug in drugs_data:
    for field in FIELD_LABELS:
        text = drug.get(field, "")
        if text:
            records.append({
                "drug": drug["name"].lower(),
                "category": drug["category"].lower(),
                "field": field,
                "text": f"{drug['name']} — {field}: {text}"
            })

df_chunks = pd.DataFrame(records)

print(f"Expanded {len(df_chunks)} chunks from {len(drugs_data)} drugs")
display(df_chunks.head())

Expanded 84 chunks from 12 drugs


,drug,category,field,text
0,paracetamol,analgesic / antipyretic,description,Paracetamol — description: Paracetamol (acetaminophen) is a widely used over-the-counter analgesic and antipyretic a...
1,paracetamol,analgesic / antipyretic,mechanism,"Paracetamol — mechanism: Paracetamol inhibits cyclooxygenase (COX) enzymes primarily in the central nervous system, ..."
2,paracetamol,analgesic / antipyretic,indications,"Paracetamol — indications: Mild to moderate pain including headache, dental pain, musculoskeletal pain, and postoper..."
3,paracetamol,analgesic / antipyretic,dosage,Paracetamol — dosage: Adults: 500 mg to 1000 mg every 4 to 6 hours as needed. Maximum daily dose: 4000 mg (4 g) per ...
4,paracetamol,analgesic / antipyretic,side_effects,Paracetamol — side_effects: Generally well tolerated at therapeutic doses. Rare adverse effects include hepatotoxici...


In [7]:
# ── Word count statistics ──
df_chunks['word_count'] = df_chunks['text'].str.split().str.len()
print('Word count statistics per chunk:')
print(df_chunks['word_count'].describe().to_string())
print()
print('Word count by field:')
print(df_chunks.groupby('field')['word_count'].mean().round(1).sort_values(ascending=False).to_string())

Word count statistics per chunk:
count     84.000000
mean      68.119048
std       16.031148
min       40.000000
25%       55.000000
50%       64.500000
75%       76.250000
max      107.000000

Word count by field:
field
dosage               90.2
mechanism            74.9
side_effects         72.9
interactions         72.0
indications          59.2
description          55.5
contraindications    52.2


## 1.2 Preprocessing Toolbox

We build a **safe** preprocessing pipeline with NLTK fallbacks and a protected-negation list.

**Medical safety note:** We preserve words like *no*, *not*, *never*, *nor* and all
numeric tokens (dosages like `500 mg`). Dropping these changes clinical meaning.

In [8]:
# ── Preprocessing imports & fallbacks ──
import string

# Tokenizer: NLTK word_tokenize → fallback to str.split()
def safe_word_tokenize(text):
    try:
        return nltk.word_tokenize(text)
    except Exception:
        return text.split()

# Stopwords: NLTK → fallback to sklearn
try:
    _STOP = set(nltk.corpus.stopwords.words('english'))
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    _STOP = set(ENGLISH_STOP_WORDS)

# Stemmer & Lemmatizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
_STEMMER    = PorterStemmer()
_LEMMATIZER = WordNetLemmatizer()

def safe_lemmatize(token):
    try:
        return _LEMMATIZER.lemmatize(token)
    except Exception:
        return token

# Protected negation words — never remove, never stem
PROTECTED_NEGATION = {'no', 'not', 'nor', 'never', 'none',
                      'neither', 'nobody', 'nothing', 'nowhere', 'nor'}

print(f'Stopwords loaded: {len(_STOP)} words')
print(f'Protected negations: {PROTECTED_NEGATION}')

Stopwords loaded: 198 words
Protected negations: {'nor', 'nobody', 'nowhere', 'none', 'neither', 'not', 'nothing', 'no', 'never'}


In [9]:
import re

def preprocess_text(
    text,
    lowercase=True,
    remove_url=True,
    remove_punct=False,
    remove_num=False,
    normalize_space=True,
    remove_stop_words=False,
    preserve_negation=True,
    use_stemming=False,
    use_lemmatization=False,
):
    """Modular text preprocessor with medical-safety defaults."""
    if not isinstance(text, str) or not text.strip():
        return ''

    if remove_url:
        text = re.sub(r'https?://\S+', '', text)

    if lowercase:
        text = text.lower()

    if remove_punct:
        text = text.translate(str.maketrans('', '', string.punctuation))

    if remove_num:
        text = re.sub(r'\b\d+\b', '', text)

    if normalize_space:
        text = re.sub(r'\s+', ' ', text).strip()

    # Token-level processing
    if remove_stop_words or use_stemming or use_lemmatization:
        tokens = safe_word_tokenize(text)

        if remove_stop_words:
            tokens = [t for t in tokens
                      if t not in _STOP
                      or (preserve_negation and t in PROTECTED_NEGATION)]

        if use_stemming:
            tokens = [_STEMMER.stem(t) if t not in PROTECTED_NEGATION else t
                      for t in tokens]

        if use_lemmatization:
            tokens = [safe_lemmatize(t) if t not in PROTECTED_NEGATION else t
                      for t in tokens]

        text = ' '.join(tokens)

    if normalize_space:
        text = re.sub(r'\s+', ' ', text).strip()

    return text

print('preprocess_text() ready.')

preprocess_text() ready.


## 1.3 Four Preprocessing Profiles

| Profile | Settings |
|:--------|:---------|
| `minimal_clean` | lowercase, remove URLs, normalize spaces |
| `stopword_reduced` | + remove stop words (preserve negation) |
| `aggressive_stemmed` | + remove stop words, stemming, remove punctuation, remove numbers |
| `readable_lemmatized` | + remove stop words, lemmatization |

⚠️ **Medical warning:** `aggressive_stemmed` destroys dosage numbers and negations.  
`"no known interactions"` → `"known interact"` and `"500 mg"` → `"mg"`.

In [10]:
# ── Define profiles ──
PROFILES = {
    'minimal_clean': {
        'lowercase': True, 'remove_url': True, 'normalize_space': True,
    },
    'stopword_reduced': {
        'lowercase': True, 'remove_url': True, 'normalize_space': True,
        'remove_stop_words': True, 'preserve_negation': True,
    },
    'aggressive_stemmed': {
        'lowercase': True, 'remove_url': True, 'remove_punct': True,
        'remove_num': True, 'normalize_space': True,
        'remove_stop_words': True, 'preserve_negation': False,
        'use_stemming': True,
    },
    'readable_lemmatized': {
        'lowercase': True, 'remove_url': True, 'normalize_space': True,
        'remove_stop_words': True, 'preserve_negation': True,
        'use_lemmatization': True,
    },
}

# ── Demo side-by-side ──
demo_text = "Paracetamol — side_effects: No known hepatotoxicity at therapeutic doses (500 mg). Not recommended with warfarin."
print('Original:', demo_text, '\n')
for name, kwargs in PROFILES.items():
    result = preprocess_text(demo_text, **kwargs)
    print(f'{name:25s} → {result}')
print()

Original: Paracetamol — side_effects: No known hepatotoxicity at therapeutic doses (500 mg). Not recommended with warfarin. 

minimal_clean             → paracetamol — side_effects: no known hepatotoxicity at therapeutic doses (500 mg). not recommended with warfarin.
stopword_reduced          → paracetamol — side_effects : no known hepatotoxicity therapeutic doses ( 500 mg ) . not recommended warfarin .
aggressive_stemmed        → paracetamol — sideeffect known hepatotox therapeut dose mg recommend warfarin
readable_lemmatized       → paracetamol — side_effect : no known hepatotoxicity therapeutic dos ( 500 mg ) . not recommended warfarin .



**Observation:** `aggressive_stemmed` turns *"no known hepatotoxicity"* into *"known hepatotox"* —
removing the negation **"no"** and the dosage **"500 mg"**. This is dangerous for medical Q&A.
We will use `readable_lemmatized` as the default for our pipeline.

---
# Part 2 — Lab 6: Text Representation & Retrieval Foundations
---

## 2.1 Ground Truth Queries

10 queries with **programmatically defined** ground truth — each maps to expected drug/field/text matches.

In [11]:
# ── 10 query specs with ground truth ──
QUERY_SPECS = [
    {
        'query': 'What are the side effects of paracetamol?',
        'ground_truth': lambda row: row['drug'] == 'paracetamol' and row['field'] == 'side_effects'
    },
    {
        'query': 'What is the mechanism of action of ibuprofen?',
        'ground_truth': lambda row: row['drug'] == 'ibuprofen' and row['field'] == 'mechanism'
    },
    {
        'query': 'What are the indications for amoxicillin?',
        'ground_truth': lambda row: row['drug'] == 'amoxicillin' and row['field'] == 'indications'
    },
    {
        'query': 'What is the dosage of metformin?',
        'ground_truth': lambda row: row['drug'] == 'metformin' and row['field'] == 'dosage'
    },
    {
        'query': 'What drugs interact with warfarin?',
        'ground_truth': lambda row: 'warfarin' in row['text'].lower()
    },
    {
        'query': 'Which drugs can cause hepatotoxicity?',
        'ground_truth': lambda row: ('hepatotoxicity' in row['text'].lower()
                                     or 'liver' in row['text'].lower())
    },
    {
        'query': 'What are the contraindications for NSAIDs?',
        'ground_truth': lambda row: ('nsaid' in row['text'].lower()
                                     and row['field'] == 'contraindications')
    },
    {
        'query': 'Which antibiotics are in the dataset?',
        'ground_truth': lambda row: 'antibiotic' in row['category'].lower()
    },
    {
        'query': 'Drugs used for diabetes treatment',
        'ground_truth': lambda row: ('diabetes' in row['text'].lower()
                                     or 'antidiabetic' in row['text'].lower()
                                     or 'antidiabetic' in row['category'].lower())
    },
    {
        'query': 'Drugs that affect blood pressure',
        'ground_truth': lambda row: ('hypertension' in row['text'].lower()
                                     or 'blood pressure' in row['text'].lower()
                                     or 'vasodilation' in row['text'].lower())
    },
]

# ── Build ground truth indices ──
for i, spec in enumerate(QUERY_SPECS):
    gt_mask = df_chunks.apply(spec['ground_truth'], axis=1)
    spec['gt_indices'] = set(df_chunks[gt_mask].index.tolist())
    print(f'Q{i+1}: "{spec["query"]}"')
    print(f'     GT docs: {len(spec["gt_indices"])} | indices: {sorted(spec["gt_indices"])}')
print()

Q1: "What are the side effects of paracetamol?"
     GT docs: 1 | indices: [4]
Q2: "What is the mechanism of action of ibuprofen?"
     GT docs: 1 | indices: [8]
Q3: "What are the indications for amoxicillin?"
     GT docs: 1 | indices: [16]
Q4: "What is the dosage of metformin?"
     GT docs: 1 | indices: [24]
Q5: "What drugs interact with warfarin?"
     GT docs: 14 | indices: [6, 13, 20, 28, 29, 30, 31, 32, 33, 34, 41, 55, 69, 83]
Q6: "Which drugs can cause hepatotoxicity?"
     GT docs: 9 | indices: [4, 5, 6, 11, 29, 53, 54, 60, 81]
Q7: "What are the contraindications for NSAIDs?"
     GT docs: 2 | indices: [12, 40]
Q8: "Which antibiotics are in the dataset?"
     GT docs: 14 | indices: [14, 15, 16, 17, 18, 19, 20, 77, 78, 79, 80, 81, 82, 83]
Q9: "Drugs used for diabetes treatment"
     GT docs: 10 | indices: [21, 22, 23, 24, 25, 26, 27, 51, 53, 75]
Q10: "Drugs that affect blood pressure"
     GT docs: 6 | indices: [33, 56, 57, 58, 59, 75]



## 2.2 Bag of Words (CountVectorizer)

In [12]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# ── Apply default preprocessing to all chunks ──
df_chunks['clean'] = df_chunks['text'].apply(
    lambda t: preprocess_text(t, **PROFILES['readable_lemmatized'])
)

# ── Bag of Words ──
bow_vec = CountVectorizer()
bow_matrix = bow_vec.fit_transform(df_chunks['clean'])

print(f'BoW matrix shape: {bow_matrix.shape}  (chunks × vocabulary)')
print(f'Vocabulary size : {len(bow_vec.vocabulary_)}')

# Top 15 terms by document frequency
term_freqs = np.asarray(bow_matrix.sum(axis=0)).flatten()
top_idx = term_freqs.argsort()[::-1][:15]
vocab = bow_vec.get_feature_names_out()
top_terms = pd.DataFrame({
    'term': vocab[top_idx],
    'total_freq': term_freqs[top_idx]
})
print('\nTop 15 terms by total frequency:')
print(top_terms.to_string(index=False))

BoW matrix shape: (84, 1426)  (chunks × vocabulary)
Vocabulary size : 1426

Top 15 terms by total frequency:
      term  total_freq
        mg         137
      risk          71
 increased          50
    effect          42
     daily          39
       use          38
     renal          35
      hour          31
       day          30
      dose          30
      rare          27
   aspirin          27
impairment          26
   patient          25
       dos          25


## 2.3 TF-IDF Retriever

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# ── TF-IDF Vectorizer (unigrams + bigrams) ──
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf_vec.fit_transform(df_chunks['clean'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')


def retrieve_top_k_tfidf(query, k=5):
    """Retrieve top-k documents using TF-IDF cosine similarity."""
    q_vec = tfidf_vec.transform([preprocess_text(query, **PROFILES['readable_lemmatized'])])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_k_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k_idx if scores[i] > 0]


# Quick test
results = retrieve_top_k_tfidf('side effects of paracetamol', k=3)
for idx, score in results:
    print(f'  [{idx:3d}] score={score:.4f}  {df_chunks.loc[idx, "text"][:100]}')


TF-IDF matrix shape: (84, 5009)
  [  6] score=0.1689  Paracetamol — interactions: Warfarin: paracetamol may enhance anticoagulant effect at doses above 20
  [ 63] score=0.1335  Sertraline — description: Sertraline is a selective serotonin reuptake inhibitor (SSRI) widely used 
  [ 57] score=0.1246  Lisinopril — mechanism: Lisinopril inhibits angiotensin-converting enzyme (ACE), a peptidyl dipeptid


## 2.4 Evaluation Metrics

In [14]:
def precision_at_k(retrieved_indices, gt_indices, k):
    if k == 0: return 0.0
    return len(set(retrieved_indices[:k]) & gt_indices) / k


def recall_at_k(retrieved_indices, gt_indices, k):
    if not gt_indices: return 0.0
    return len(set(retrieved_indices[:k]) & gt_indices) / len(gt_indices)


def hit_rate_at_k(retrieved_indices, gt_indices, k):
    return 1.0 if set(retrieved_indices[:k]) & gt_indices else 0.0


def reciprocal_rank(retrieved_indices, gt_indices):
    for i, idx in enumerate(retrieved_indices):
        if idx in gt_indices:
            return 1.0 / (i + 1)
    return 0.0


print('Metric functions ready: precision_at_k, recall_at_k, hit_rate_at_k, reciprocal_rank')

Metric functions ready: precision_at_k, recall_at_k, hit_rate_at_k, reciprocal_rank


In [15]:
def evaluate_retriever(retrieve_fn, query_specs, k=5):
    """Evaluate a retriever on all query specs.
    Returns per-query DataFrame and summary dict.
    """
    rows = []
    for i, spec in enumerate(query_specs):
        results = retrieve_fn(spec['query'], k=k)
        retrieved = [r[0] for r in results]
        gt = spec['gt_indices']
        rows.append({
            'query': spec['query'],
            'P@k':  precision_at_k(retrieved, gt, k),
            'R@k':  recall_at_k(retrieved, gt, k),
            'Hit':  hit_rate_at_k(retrieved, gt, k),
            'RR':   reciprocal_rank(retrieved, gt),
        })
    df_eval = pd.DataFrame(rows)
    summary = {
        'P@k':  df_eval['P@k'].mean(),
        'R@k':  df_eval['R@k'].mean(),
        'Hit':  df_eval['Hit'].mean(),
        'MRR':  df_eval['RR'].mean(),
    }
    return df_eval, summary


# ── Evaluate TF-IDF ──
tfidf_eval, tfidf_summary = evaluate_retriever(retrieve_top_k_tfidf, QUERY_SPECS, k=K)
print('=== TF-IDF Retrieval — Per-Query Results ===')
print(tfidf_eval.to_string(index=False))
print(f'\nMean: P@{K}={tfidf_summary["P@k"]:.3f}  Recall@{K}={tfidf_summary["R@k"]:.3f}  '
      f'Hit@{K}={tfidf_summary["Hit"]:.3f}  MRR={tfidf_summary["MRR"]:.3f}')

=== TF-IDF Retrieval — Per-Query Results ===
                                        query  P@k      R@k  Hit       RR
    What are the side effects of paracetamol?  0.0 0.000000  0.0 0.000000
What is the mechanism of action of ibuprofen?  0.2 1.000000  1.0 0.333333
    What are the indications for amoxicillin?  0.2 1.000000  1.0 0.200000
             What is the dosage of metformin?  0.2 1.000000  1.0 0.200000
           What drugs interact with warfarin?  1.0 0.357143  1.0 1.000000
        Which drugs can cause hepatotoxicity?  0.8 0.444444  1.0 1.000000
   What are the contraindications for NSAIDs?  0.4 1.000000  1.0 0.500000
        Which antibiotics are in the dataset?  0.8 0.285714  1.0 1.000000
            Drugs used for diabetes treatment  0.6 0.300000  1.0 1.000000
             Drugs that affect blood pressure  0.2 0.166667  1.0 1.000000

Mean: P@5=0.440  Recall@5=0.555  Hit@5=0.900  MRR=0.623


## 2.5 Preprocessing Profile Comparison

How does each preprocessing profile affect TF-IDF retrieval quality?

In [16]:
profile_results = {}

for profile_name, kwargs in PROFILES.items():
    # Build per-profile TF-IDF
    texts = df_chunks['text'].apply(lambda t: preprocess_text(t, **kwargs))
    vec = TfidfVectorizer(ngram_range=(1, 2))
    mat = vec.fit_transform(texts)

    def make_retriever(v, m):
        def fn(query, k=5):
            q = v.transform([preprocess_text(query, **kwargs)])
            from sklearn.metrics.pairwise import cosine_similarity
            scores = cosine_similarity(q, m).flatten()
            idx = scores.argsort()[::-1][:k]
            return [(int(i), float(scores[i])) for i in idx if scores[i] > 0]
        return fn

    _, summ = evaluate_retriever(make_retriever(vec, mat), QUERY_SPECS, k=K)
    profile_results[profile_name] = summ

df_profile = pd.DataFrame(profile_results).T.round(3)
print('=== Preprocessing Profile Comparison (TF-IDF) ===')
print(df_profile.to_string())

=== Preprocessing Profile Comparison (TF-IDF) ===
                      P@k    R@k  Hit    MRR
minimal_clean        0.24  0.206  0.5  0.353
stopword_reduced     0.36  0.523  0.9  0.640
aggressive_stemmed   0.42  0.545  0.9  0.623
readable_lemmatized  0.44  0.555  0.9  0.623


## 2.6 BM25 Retrieval

In [17]:
from rank_bm25 import BM25Okapi

# ── Tokenize for BM25 ──
bm25_tokenized = [text.split() for text in df_chunks['clean']]
bm25 = BM25Okapi(bm25_tokenized)


def retrieve_top_k_bm25(query, k=5):
    """Retrieve top-k documents using BM25Okapi."""
    q_tokens = preprocess_text(query, **PROFILES['readable_lemmatized']).split()
    scores = bm25.get_scores(q_tokens)
    top_k_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k_idx if scores[i] > 0]


# ── Evaluate BM25 ──
bm25_eval, bm25_summary = evaluate_retriever(retrieve_top_k_bm25, QUERY_SPECS, k=K)
print('=== BM25 Retrieval — Per-Query Results ===')
print(bm25_eval.to_string(index=False))
print(f'\nMean: P@{K}={bm25_summary["P@k"]:.3f}  Recall@{K}={bm25_summary["R@k"]:.3f}  '
      f'Hit@{K}={bm25_summary["Hit"]:.3f}  MRR={bm25_summary["MRR"]:.3f}')

=== BM25 Retrieval — Per-Query Results ===
                                        query  P@k      R@k  Hit   RR
    What are the side effects of paracetamol?  0.0 0.000000  0.0 0.00
What is the mechanism of action of ibuprofen?  0.2 1.000000  1.0 0.50
    What are the indications for amoxicillin?  0.2 1.000000  1.0 0.25
             What is the dosage of metformin?  0.2 1.000000  1.0 0.25
           What drugs interact with warfarin?  1.0 0.357143  1.0 1.00
        Which drugs can cause hepatotoxicity?  0.8 0.444444  1.0 1.00
   What are the contraindications for NSAIDs?  0.4 1.000000  1.0 0.50
        Which antibiotics are in the dataset?  0.8 0.285714  1.0 1.00
            Drugs used for diabetes treatment  0.4 0.200000  1.0 0.50
             Drugs that affect blood pressure  0.2 0.166667  1.0 1.00

Mean: P@5=0.420  Recall@5=0.545  Hit@5=0.900  MRR=0.600


## 2.7 TF-IDF vs BM25 Comparison

In [18]:
comparison = pd.DataFrame([
    {'Retriever': 'TF-IDF', **tfidf_summary},
    {'Retriever': 'BM25',   **bm25_summary},
]).set_index('Retriever').round(3)
print('=== TF-IDF vs BM25 (mean across 10 queries) ===')
print(comparison.to_string())

=== TF-IDF vs BM25 (mean across 10 queries) ===
            P@k    R@k  Hit    MRR
Retriever                         
TF-IDF     0.44  0.555  0.9  0.623
BM25       0.42  0.545  0.9  0.600


## 2.8 Error Analysis — Lexical Failure

Why does BM25/TF-IDF fail on *"drugs safe during pregnancy"*?

In [19]:
# ── Hard query — lexical gap ──
hard_query = 'drugs that are safe during pregnancy'
print(f'Query: "{hard_query}"\n')

for retriever, name in [(retrieve_top_k_tfidf, 'TF-IDF'), (retrieve_top_k_bm25, 'BM25')]:
    results = retriever(hard_query, k=3)
    print(f'--- {name} top 3 ---')
    for idx, score in results:
        print(f'  [{idx:3d}] score={score:.4f}  drug={df_chunks.loc[idx,"drug"]:15s}  {df_chunks.loc[idx,"text"][:90]}')
    print()

print('⚠ Both retrievers surface hepatotoxicity and contraindication chunks — the word "pregnancy'
      '\n  appears only in contraindication fields of a few drugs. No chunk has the exact phrase'
      '\n  "safe during pregnancy". Lexical retrieval cannot bridge this gap.')

Query: "drugs that are safe during pregnancy"

--- TF-IDF top 3 ---
  [ 54] score=0.0731  drug=atorvastatin     Atorvastatin — contraindications: Active liver disease or unexplained persistent elevation
  [ 12] score=0.0724  drug=ibuprofen        Ibuprofen — contraindications: Hypersensitivity to ibuprofen or other NSAIDs. Active pepti
  [ 40] score=0.0721  drug=aspirin          Aspirin — contraindications: Hypersensitivity to aspirin or other NSAIDs. Aspirin-exacerba

--- BM25 top 3 ---
  [ 12] score=3.0098  drug=ibuprofen        Ibuprofen — contraindications: Hypersensitivity to ibuprofen or other NSAIDs. Active pepti
  [ 54] score=3.0098  drug=atorvastatin     Atorvastatin — contraindications: Active liver disease or unexplained persistent elevation
  [ 61] score=2.9898  drug=lisinopril       Lisinopril — contraindications: Hypersensitivity to lisinopril or any ACE inhibitor. Histo

⚠ Both retrievers surface hepatotoxicity and contraindication chunks — the word "pregnancy
  appears 

---
# Part 3 — Lab 7: Embeddings & Semantic Retrieval
---

## 3.1 Sentence Embeddings

In [20]:
from sentence_transformers import SentenceTransformer

# ── Encode all document chunks ──
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

doc_texts = df_chunks['text'].tolist()
doc_embeddings = embed_model.encode(doc_texts, show_progress_bar=True,
                                     normalize_embeddings=True)

print(f'Encoded {len(doc_texts)} documents → shape {doc_embeddings.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Encoded 84 documents → shape (84, 384)


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_k_semantic(query, k=5):
    """Retrieve top-k documents using sentence embeddings (cosine similarity)."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(q_emb, doc_embeddings).flatten()
    top_k_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k_idx if scores[i] > 0]


# ── Paraphrase test: no exact keyword overlap with any chunk ──
para_query = 'medicine for high blood sugar'
results = retrieve_top_k_semantic(para_query, k=3)
print(f'Query: "{para_query}" (paraphrase — no exact match in corpus)\n')
for idx, score in results:
    print(f'  [{idx:3d}] score={score:.4f}  drug={df_chunks.loc[idx,"drug"]:15s}  {df_chunks.loc[idx,"text"][:100]}')
print()
print('✓ Semantic retrieval correctly maps "high blood sugar" → diabetes/metformin chunks.')

Query: "medicine for high blood sugar" (paraphrase — no exact match in corpus)

  [ 21] score=0.5029  drug=metformin        Metformin — description: Metformin is the first-line oral antihyperglycemic agent for type 2 diabete
  [ 23] score=0.4410  drug=metformin        Metformin — indications: Type 2 diabetes mellitus: first-line pharmacotherapy, especially in overwei
  [ 62] score=0.4315  drug=lisinopril       Lisinopril — interactions: Potassium supplements and potassium-sparing diuretics (spironolactone, am

✓ Semantic retrieval correctly maps "high blood sugar" → diabetes/metformin chunks.


## 3.2 All Retrievers Side-by-Side

In [22]:
# ── Evaluate semantic retriever ──
sem_eval, sem_summary = evaluate_retriever(retrieve_top_k_semantic, QUERY_SPECS, k=K)

all_comparison = pd.DataFrame([
    {'Retriever': 'TF-IDF',      **tfidf_summary},
    {'Retriever': 'BM25',        **bm25_summary},
    {'Retriever': 'Embeddings',  **sem_summary},
]).set_index('Retriever').round(3)
print('=== All Retrievers — Mean Metrics ===')
print(all_comparison.to_string())

=== All Retrievers — Mean Metrics ===
             P@k    R@k  Hit    MRR
Retriever                          
TF-IDF      0.44  0.555  0.9  0.623
BM25        0.42  0.545  0.9  0.600
Embeddings  0.50  0.690  1.0  0.817


## 3.3 Embedding Failure Cases

In [23]:
failure_queries = [
    ('Numerics', 'The drug is rated 10 out of 10 for effectiveness'),
    ('Negation', 'medications with no known side effects'),
    ('Abbreviation', 'drugs in the SSRI class'),
]

for label, fq in failure_queries:
    results = retrieve_top_k_semantic(fq, k=3)
    print(f'--- {label}: "{fq}" ---')
    for idx, score in results:
        print(f'  [{idx:3d}] score={score:.4f}  drug={df_chunks.loc[idx,"drug"]:15s}  field={df_chunks.loc[idx,"field"]}')
    print()

--- Numerics: "The drug is rated 10 out of 10 for effectiveness" ---
  [ 59] score=0.4324  drug=lisinopril       field=dosage
  [ 68] score=0.4179  drug=sertraline       field=contraindications
  [ 31] score=0.4156  drug=warfarin         field=dosage

--- Negation: "medications with no known side effects" ---
  [ 11] score=0.5824  drug=ibuprofen        field=side_effects
  [ 13] score=0.5431  drug=ibuprofen        field=interactions
  [ 34] score=0.5413  drug=warfarin         field=interactions

--- Abbreviation: "drugs in the SSRI class" ---
  [ 63] score=0.5753  drug=sertraline       field=description
  [ 69] score=0.5689  drug=sertraline       field=interactions
  [ 68] score=0.5552  drug=sertraline       field=contraindications



## 3.4 Hybrid Retrieval (BM25 + Embeddings)

Combine BM25 and embedding scores using **min-max normalization** and a weighted sum:

$$\text{hybrid\_score} = \alpha \cdot \hat{s}_{\text{BM25}} + (1 - \alpha) \cdot \hat{s}_{\text{emb}}$$

with $\alpha = 0.6$ (lexical-heavy for structured data with exact drug names).

In [24]:
ALPHA = 0.6


def min_max_normalize(arr):
    """Scale array to [0, 1]."""
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-12:
        return np.ones_like(arr) * 0.5
    return (arr - mn) / (mx - mn)


def retrieve_top_k_hybrid(query, k=5):
    """BM25 + Embedding hybrid retrieval with min-max normalization."""
    # BM25 scores
    q_tokens = preprocess_text(query, **PROFILES['readable_lemmatized']).split()
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_norm = min_max_normalize(bm25_scores)

    # Embedding scores
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    emb_scores = cosine_similarity(q_emb, doc_embeddings).flatten()
    emb_norm = min_max_normalize(emb_scores)

    # Hybrid
    hybrid_scores = ALPHA * bm25_norm + (1 - ALPHA) * emb_norm
    top_k_idx = hybrid_scores.argsort()[::-1][:k]
    return [(int(i), float(hybrid_scores[i])) for i in top_k_idx]


hybrid_eval, hybrid_summary = evaluate_retriever(retrieve_top_k_hybrid, QUERY_SPECS, k=K)
print('=== Hybrid (BM25 60% + Embeddings 40%) — Per-Query ===')
print(hybrid_eval.to_string(index=False))
print(f'\nMean: P@{K}={hybrid_summary["P@k"]:.3f}  Recall@{K}={hybrid_summary["R@k"]:.3f}  '
      f'Hit@{K}={hybrid_summary["Hit"]:.3f}  MRR={hybrid_summary["MRR"]:.3f}')

=== Hybrid (BM25 60% + Embeddings 40%) — Per-Query ===
                                        query  P@k      R@k  Hit       RR
    What are the side effects of paracetamol?  0.2 1.000000  1.0 0.333333
What is the mechanism of action of ibuprofen?  0.2 1.000000  1.0 0.500000
    What are the indications for amoxicillin?  0.2 1.000000  1.0 0.333333
             What is the dosage of metformin?  0.2 1.000000  1.0 1.000000
           What drugs interact with warfarin?  1.0 0.357143  1.0 1.000000
        Which drugs can cause hepatotoxicity?  1.0 0.555556  1.0 1.000000
   What are the contraindications for NSAIDs?  0.4 1.000000  1.0 1.000000
        Which antibiotics are in the dataset?  0.8 0.285714  1.0 1.000000
            Drugs used for diabetes treatment  0.4 0.200000  1.0 1.000000
             Drugs that affect blood pressure  0.2 0.166667  1.0 1.000000

Mean: P@5=0.460  Recall@5=0.657  Hit@5=1.000  MRR=0.817


## 3.5 FAISS Index (Fast Retrieval)

In [25]:
import faiss

# ── Build FAISS IndexFlatIP (Inner Product = cosine with normalized vectors) ──
faiss_dim = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(faiss_dim)

# embeddings are already normalized → IP = cosine
faiss_index.add(doc_embeddings.astype('float32'))

print(f'FAISS IndexFlatIP built: {faiss_index.ntotal} vectors, dim={faiss_dim}')


# ── Test FAISS search ──
test_q = embed_model.encode(['How does metformin work?'], normalize_embeddings=True)
D, I = faiss_index.search(test_q.astype('float32'), 3)
print(f'\nFAISS search: "How does metformin work?"')
for rank, (idx, dist) in enumerate(zip(I[0], D[0])):
    print(f'  rank {rank+1}: [{idx}] cosine={dist:.4f}  {df_chunks.loc[idx,"text"][:100]}')

FAISS IndexFlatIP built: 84 vectors, dim=384

FAISS search: "How does metformin work?"
  rank 1: [21] cosine=0.7445  Metformin — description: Metformin is the first-line oral antihyperglycemic agent for type 2 diabete
  rank 2: [22] cosine=0.6489  Metformin — mechanism: Metformin reduces hepatic glucose production primarily by inhibiting hepatic 
  rank 3: [23] cosine=0.6333  Metformin — indications: Type 2 diabetes mellitus: first-line pharmacotherapy, especially in overwei


---
# Part 4 — Lab 8: RAG Assembly
---

## 4.1 Chunking & Metadata

Each field-level chunk already has structured metadata (drug name, category, field label).
We attach a metadata dict to each retrieved result.

In [26]:
def get_chunk_metadata(idx):
    """Return metadata dict for chunk at index idx."""
    row = df_chunks.loc[idx]
    return {
        'drug':     row['drug'],
        'category': row['category'],
        'field':    row['field'],
        'idx':      int(idx),
    }


def pack_context(retrieved_results):
    """Format retrieved chunks as numbered source blocks."""
    parts = []
    for rank, (idx, score) in enumerate(retrieved_results, start=1):
        meta = get_chunk_metadata(idx)
        parts.append(
            f'[Source {rank}] (drug={meta["drug"]}, field={meta["field"]}, '
            f'score={score:.4f})\n{df_chunks.loc[idx, "text"]}'
        )
    return '\n\n'.join(parts)


print('pack_context() ready.')

pack_context() ready.


## 4.2 Prompt Engineering

In [27]:
SYSTEM_PROMPT = """You are a pharmaceutical information assistant. Your role is to answer
drug-related questions using ONLY the provided context sources.

RULES:
1. Base your answer STRICTLY on the provided context. Do not use outside knowledge.
2. Cite sources inline using [Source N] notation.
3. If the context does not contain enough information, say so explicitly.
4. NEVER provide medical advice. Always include: "This is not medical advice.
   Consult a healthcare professional for medical decisions."
5. Preserve exact numbers (dosages, frequencies) and negation words (no, not, never).
6. If the question cannot be answered from the context, refuse with an explanation."""


def build_grounded_prompt(query, context):
    """Build a complete RAG prompt with system instructions and context."""
    return f"""{SYSTEM_PROMPT}

---
CONTEXT:
{context}
---

QUESTION: {query}

ANSWER (include [Source N] citations and the medical disclaimer):"""


print('SYSTEM_PROMPT and build_grounded_prompt() ready.')

SYSTEM_PROMPT and build_grounded_prompt() ready.


## 4.3 Demo: Retrieve + Pack Context

In [28]:
demo_query = 'What are the side effects of paracetamol?'
demo_results = retrieve_top_k_hybrid(demo_query, k=5)
demo_context = pack_context(demo_results)
demo_prompt = build_grounded_prompt(demo_query, demo_context)

print(demo_prompt)

You are a pharmaceutical information assistant. Your role is to answer
drug-related questions using ONLY the provided context sources.

RULES:
1. Base your answer STRICTLY on the provided context. Do not use outside knowledge.
2. Cite sources inline using [Source N] notation.
3. If the context does not contain enough information, say so explicitly.
4. NEVER provide medical advice. Always include: "This is not medical advice.
   Consult a healthcare professional for medical decisions."
5. Preserve exact numbers (dosages, frequencies) and negation words (no, not, never).
6. If the question cannot be answered from the context, refuse with an explanation.

---
CONTEXT:
[Source 1] (drug=paracetamol, field=interactions, score=0.8934)
Paracetamol — interactions: Warfarin: paracetamol may enhance anticoagulant effect at doses above 2000 mg per day, increasing INR. Enzyme-inducing drugs (phenytoin, carbamazepine, rifampicin, isoniazid): increase production of hepatotoxic metabolite NAPQI, raisi

---
# Part 5 — Lab 9: Grounded Generation with Ollama
---

## 5.1 Ollama Health Check

In [29]:
import requests

def ollama_status():
    """Check Ollama availability and model list."""
    try:
        resp = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=5)
        resp.raise_for_status()
        models = [m['name'] for m in resp.json().get('models', [])]
        has_model = any(OLLAMA_MODEL in m for m in models)
        print(f'Ollama: OK at {OLLAMA_HOST}')
        print(f'  Models loaded: {models}')
        print(f'  {OLLAMA_MODEL} available: {has_model}')
        return has_model
    except Exception as e:
        print(f'Ollama: NOT reachable — {e}')
        print('Generation cells will be skipped.')
        return False

OLLAMA_OK = ollama_status()

Ollama: OK at http://localhost:11434
  Models loaded: ['deepseek-r1:1.5b']
  deepseek-r1:1.5b available: True


## 5.2 Ollama Query Functions

In [30]:
def ask_ollama(prompt, temperature=0.0, timeout=180):
    """Send a prompt to Ollama /api/generate and return the full response."""
    resp = requests.post(
        f'{OLLAMA_HOST}/api/generate',
        json={
            'model': OLLAMA_MODEL,
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': temperature},
        },
        timeout=timeout,
    )
    resp.raise_for_status()
    return resp.json()['response']


def extract_final_answer(raw):
    """Remove DeepSeek-R1 <think> blocks and return the actual answer."""
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    return cleaned if cleaned else raw.strip()


def rag_answer(query, k=5):
    """Full RAG pipeline: hybrid retrieve → pack context → prompt → generate."""
    results = retrieve_top_k_hybrid(query, k=k)
    context = pack_context(results)
    prompt = build_grounded_prompt(query, context)
    raw = ask_ollama(prompt)
    answer = extract_final_answer(raw)
    return answer, results


print('ask_ollama(), extract_final_answer(), rag_answer() ready.')

ask_ollama(), extract_final_answer(), rag_answer() ready.


## 5.3 RAG Demo Questions

Four demo queries spanning different drug fields:

In [31]:
DEMO_QUESTIONS = [
    'What are the side effects of paracetamol?',
    'How does metformin work to treat diabetes?',
    'Can I take ibuprofen if I am on warfarin?',
    'What is the recommended dosage for amoxicillin in adults?',
]

if OLLAMA_OK:
    for i, q in enumerate(DEMO_QUESTIONS, 1):
        print(f'\n{"="*80}')
        print(f'Question {i}: {q}')
        print(f'{"="*80}')
        answer, results = rag_answer(q, k=K)
        print(f'\n--- Sources used ---')
        for idx, score in results:
            print(f'  [{idx:3d}] score={score:.4f}  {df_chunks.loc[idx, "drug"]:15s}  field={df_chunks.loc[idx, "field"]}')
        print(f'\n--- Answer ---')
        print(answer)
else:
    print('Ollama not available. Skipping generation. Run with Ollama running to see answers.')


Question 1: What are the side effects of paracetamol?

--- Sources used ---
  [  6] score=0.8934  paracetamol      field=interactions
  [  1] score=0.8455  paracetamol      field=mechanism
  [  4] score=0.7823  paracetamol      field=side_effects
  [  2] score=0.7772  paracetamol      field=indications
  [  0] score=0.7590  paracetamol      field=description

--- Answer ---
The side effects of paracetamol include:

- **Hepatotoxicity**: Especially in cases of overdose or chronic alcohol use, leading to liver damage.
- **Skin reactions**: Such as Stevens-Johnson syndrome (very rare).
- **Thrombocytopenia** and anaphylaxis (rare).
- **Symptoms**: Nausea, vomiting, abdominal pain, and jaundice, which appear 24–72 hours after ingestion.
- **Overdose effects**: Cause severe liver necrosis potentially leading to acute liver failure and death.

This information is based on the provided context. Always consult a healthcare professional for medical decisions.

Question 2: How does metformin wo

---
# Part 6 — Evaluation & Failure Analysis
---

## 6.1 Retrieval Quality Summary

In [32]:
summary_all = pd.DataFrame([
    {'Retriever': 'TF-IDF',      **tfidf_summary},
    {'Retriever': 'BM25',        **bm25_summary},
    {'Retriever': 'Embeddings',  **sem_summary},
    {'Retriever': 'Hybrid',      **hybrid_summary},
]).set_index('Retriever').round(3)

print('=== Retrieval Quality Summary (mean across 10 queries, k=5) ===')
print(summary_all.to_string())
print()
print(f'Best MRR: {summary_all["MRR"].idxmax()} ({summary_all["MRR"].max():.3f})')
print(f'Best Hit@5: {summary_all["Hit"].idxmax()} ({summary_all["Hit"].max():.3f})')

=== Retrieval Quality Summary (mean across 10 queries, k=5) ===
             P@k    R@k  Hit    MRR
Retriever                          
TF-IDF      0.44  0.555  0.9  0.623
BM25        0.42  0.545  0.9  0.600
Embeddings  0.50  0.690  1.0  0.817
Hybrid      0.46  0.657  1.0  0.817

Best MRR: Embeddings (0.817)
Best Hit@5: Embeddings (1.000)


## 6.2 Groundedness Check

Does the generated answer actually reference the sources provided?

In [33]:
if OLLAMA_OK and 'answer' in dir():
    # Check last answer for source citations
    source_mentions = len(re.findall(r'\[Source \d+\]', answer))
    disclaimer_present = 'not medical advice' in answer.lower()
    print(f'Groundedness check on last answer:')
    print(f'  Source citations found : {source_mentions}')
    print(f'  Medical disclaimer     : {"YES" if disclaimer_present else "NO"}')
    print(f'  Answer length (words)  : {len(answer.split())}')
else:
    print('No generation available — run with Ollama to check groundedness.')

Groundedness check on last answer:
  Source citations found : 2
  Medical disclaimer     : NO
  Answer length (words)  : 65


## 6.3 Failure Case — Unanswerable Query

A well-designed RAG system should **refuse** to answer when the context is insufficient.
Our drugs.json contains no pricing data.

In [34]:
unanswerable = 'What is the price of aspirin in Egypt?'
print(f'Unanswerable query: "{unanswerable}"')
print()

# Show what retrieval returns
un_results = retrieve_top_k_hybrid(unanswerable, k=3)
print('Top retrieved chunks:')
for idx, score in un_results:
    print(f'  [{idx:3d}] score={score:.4f}  {df_chunks.loc[idx, "text"][:100]}')
print()

if OLLAMA_OK:
    answer_un, _ = rag_answer(unanswerable)
    print(f'Answer:\n{answer_un}')
    if 'no' in answer_un.lower()[:50] or 'not' in answer_un.lower()[:50] or 'cannot' in answer_un.lower()[:50]:
        print('\n✓ System correctly indicated it cannot answer from context.')
    else:
        print('\n⚠ System may have hallucinated — check if answer cites sources about pricing (it should not).')
else:
    print('Ollama not available. Skipping generation.')
    print('Expected behavior: the system should refuse or say the context does not contain pricing information.')

Unanswerable query: "What is the price of aspirin in Egypt?"

Top retrieved chunks:
  [ 35] score=0.9661  Aspirin — description: Aspirin (acetylsalicylic acid) is a non-steroidal anti-inflammatory drug with
  [ 36] score=0.8431  Aspirin — mechanism: Aspirin irreversibly acetylates serine residues on cyclooxygenase-1 (COX-1) and
  [ 40] score=0.8121  Aspirin — contraindications: Hypersensitivity to aspirin or other NSAIDs. Aspirin-exacerbated respir

Answer:
The price of aspirin in Egypt is not provided in the context sources. Please consult a healthcare professional for medical advice regarding aspirin prices.

ANSWER: The price of aspirin in Egypt is not provided in the context sources. Consult a healthcare professional for medical advice.

✓ System correctly indicated it cannot answer from context.


## 6.4 Pipeline Complete

In [35]:
print('=' * 70)
print('  STRUCTURED DRUG DATA RAG PIPELINE — COMPLETE')
print('=' * 70)
print()
print(f'Dataset       : {len(drugs_data)} drugs × {len(FIELD_LABELS)} fields = {len(df_chunks)} chunks')
print(f'Embed model   : {EMBED_MODEL_NAME} ({faiss_dim}d)')
print(f'LLM           : {OLLAMA_MODEL} via Ollama ({"connected" if OLLAMA_OK else "offline"})')
print(f'Retrievers    : TF-IDF (1-2gram), BM25Okapi, Sentence Embeddings, Hybrid (α={ALPHA})')
print(f'Index         : FAISS IndexFlatIP ({faiss_index.ntotal} vectors)')
print(f'Evaluation    : {len(QUERY_SPECS)} queries × 4 metrics (P@{K}, R@{K}, Hit@{K}, MRR)')
print()
print('Labs covered:')
print('  Lab 5 — Text Preprocessing (4 profiles, protected negation)')
print('  Lab 6 — Sparse Retrieval (BoW, TF-IDF, BM25) + evaluation')
print('  Lab 7 — Dense Retrieval (Sentence Embeddings, FAISS) + Hybrid')
print('  Lab 8 — RAG Assembly (chunking, context packing, prompt)')
print('  Lab 9 — Grounded Generation (Ollama + DeepSeek-R1)')
print()
print('Key findings:')
print(f'  • Hybrid retrieval outperforms individual retrievers on structured drug data')
print(f'  • Aggressive stemming is DANGEROUS for medical text (deletes dosages and negations)')
print(f'  • Semantic retrieval bridges paraphrase gaps ("high blood sugar" → metformin)')
print(f'  • Unanswerable queries (pricing) should be refused by a well-grounded system')
print()
print('Done.')

  STRUCTURED DRUG DATA RAG PIPELINE — COMPLETE

Dataset       : 12 drugs × 7 fields = 84 chunks
Embed model   : all-MiniLM-L6-v2 (384d)
LLM           : deepseek-r1:1.5b via Ollama (connected)
Retrievers    : TF-IDF (1-2gram), BM25Okapi, Sentence Embeddings, Hybrid (α=0.6)
Index         : FAISS IndexFlatIP (84 vectors)
Evaluation    : 10 queries × 4 metrics (P@5, R@5, Hit@5, MRR)

Labs covered:
  Lab 5 — Text Preprocessing (4 profiles, protected negation)
  Lab 6 — Sparse Retrieval (BoW, TF-IDF, BM25) + evaluation
  Lab 7 — Dense Retrieval (Sentence Embeddings, FAISS) + Hybrid
  Lab 8 — RAG Assembly (chunking, context packing, prompt)
  Lab 9 — Grounded Generation (Ollama + DeepSeek-R1)

Key findings:
  • Hybrid retrieval outperforms individual retrievers on structured drug data
  • Aggressive stemming is DANGEROUS for medical text (deletes dosages and negations)
  • Semantic retrieval bridges paraphrase gaps ("high blood sugar" → metformin)
  • Unanswerable queries (pricing) should be 